# Archived experiment source: EndekGAN_T2I_Master_Experiment_Recap.ipynb
All historical cell outputs were cleared for repository publication. The original SHA-256 is recorded in docs/SOURCE_MANIFEST.json.
WARNING: direct Run All may train models, download dependencies, or create files. Use the master orchestrator first. The public folder excludes copyrighted source images and model weights.


# EndekGAN-T2I — Master Experiment Recap and Evidence Audit

**Scope:** This notebook reconstructs *one integrated research study* from preserved experiments; it does **not retrain**, revise checkpoints, or treat CPU/CUDA development stages as separate research methods.

**Traceable flow:** curated Endek dataset → real-image samples → structured text attributes → research framework → 21 trained configurations → training diagnostics → quantitative evaluation → qualitative real-vs-generated comparisons → repeat-tile inspection → multilingual CLIP diagnostic and expert-assessment status → RQ1–RQ3 evidence map.

**Claim boundaries:** FID/KID on 37 held-out real images are exploratory; seam-gap measures opposing-edge pixel agreement, not ornament accuracy; prompt metadata are not expert-validated; the multilingual CLIP output available here covers only seed 42; the external DF-GAN–MiniLM uses an adapted architecture and independent training settings.

In [ ]:
from pathlib import Path
import os, json, hashlib, shutil, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Image as IPyImage
ROOT=Path(os.environ.get("ENDEKGAN_PROJECT_ROOT", "../EndekGAN-T2I")).resolve()
HERE=Path(os.environ.get("ENDEKGAN_RECAP_OUTPUT", str(ROOT/"EXPERIMENT_MASTER_RECAP_20261105"))).resolve()
FIG=HERE/"FIGURES"; TAB=HERE/"TABLES"
FIG.mkdir(parents=True,exist_ok=True); TAB.mkdir(parents=True,exist_ok=True)
INTERNAL=ROOT/"EndekGAN_IEEE_V5_OUTPUT"
RUNS=INTERNAL/"FULL_runs_CPU_bs4"
SPLITS=INTERNAL/"fixed_split"
EXTERNAL=ROOT/"EndekGAN_V6_EXTERNAL_BASELINES"
ADD=ROOT/"EVALUATION_EXTENSION_20261105"/"RESULTS"
SEEDS=(42,123,2025)
LABELS={"flora":"Flora","fauna":"Fauna","dekoratif":"Decorative","geometris":"Geometric"}
print("Working folder:",HERE)
print("The research files below are read-only sources; new tables/figures go in HERE.")

## 1. Dataset provenance, duplicate audit and fixed split

The original curation report records raw counts and exact duplicates removed. The three saved split CSV files are used as the authoritative training/validation/test partitions; no split is recomputed.

In [ ]:
required=[SPLITS/f"{s}_split.csv" for s in ("train","val","test")]+[
    INTERNAL/"E1_E7_NEW_RESULTS_BY_RUN.csv",
    EXTERNAL/"E8_EXTERNAL_MACRO_BY_RUN.csv",
    ADD/"SEED_42"/"FIG_MODEL_COMPARISON.png"]
missing=[str(p) for p in required if not p.is_file()]
assert not missing, "Missing source files: "+str(missing)
audit=json.loads((SPLITS/"dataset_audit.json").read_text(encoding="utf-8"))
split={s:pd.read_csv(SPLITS/f"{s}_split.csv") for s in ("train","val","test")}
assert [len(split[s]) for s in ("train","val","test")]==[296,37,37]
assert sum(len(v) for v in split.values())==audit["unique_md5_images"]==370
assert len(set().union(*(set(v.md5) for v in split.values())))==370
table=pd.DataFrame({s:split[s].label.value_counts() for s in ("train","val","test")}).fillna(0).astype(int)
table["total"]=table.sum(axis=1)
table=table.reindex(["flora","fauna","dekoratif","geometris"])
table.index=[LABELS[k] for k in table.index]
table.loc["TOTAL"]=table.sum(numeric_only=True)
display(table)
print("Raw:",audit["raw_images"],"| Removed exact duplicates:",audit["exact_duplicate_removed"],"| Unique images:",audit["unique_md5_images"])
print("Unique prompts:",audit["unique_prompts"],"| Expert-verified rows:",audit["expert_verified_rows"])
table.to_csv(TAB/"TABLE_01_FIXED_DATASET_SPLIT.csv",index_label="motif_class")

## 2. Original Endek-image samples — interactive category selector

The dropdown displays **only one motif class at a time** (Flora, Fauna, Decorative, or Geometric), with three deterministic training examples per class. The grayscale checkbox optionally converts the selected examples to black-and-white. Each displayed panel is saved separately under \`FIGURES\`; the original dataset is never altered. The \`sample_paths\` list remains available for preprocessing diagnostics in the next cell.

In [ ]:
# Build deterministic samples, preserving sample_paths for downstream preprocessing.
sample_paths = []
for c in ["flora", "fauna", "dekoratif", "geometris"]:
    t = split["train"].loc[split["train"].label == c].sort_values("filename").head(3)
    assert len(t) == 3, f"Expected 3 samples for {c}, found {len(t)}"
    for _, r in t.iterrows():
        p = Path(str(r.image_path))
        assert p.is_file(), p
        sample_paths.append((c, p, str(r.filename)))

def show_endek_category(category="flora", grayscale=False):
    """Display only one category at a time and export its panel."""
    chosen = [(cl, p, name) for cl, p, name in sample_paths if cl == category]
    if not chosen:
        raise ValueError(f"No samples for category: {category}")
    fig, axes = plt.subplots(1, len(chosen), figsize=(9.5, 3.7), squeeze=False)
    for ax, (cl, p, name) in zip(axes.flat, chosen):
        with Image.open(p) as img:
            motif = img.convert("L" if grayscale else "RGB").resize((256, 256))
            ax.imshow(motif, cmap="gray" if grayscale else None)
        ax.set_title(Path(name).name[:23], fontsize=9)
        ax.axis("off")
    mode = "GRAYSCALE" if grayscale else "COLOR"
    fig.suptitle(f"Real Balinese Endek — {LABELS[category]} ({mode})", fontsize=13)
    fig.tight_layout()
    file_path = FIG / f"FIG_01_REAL_ENDEK_{category.upper()}_{mode}.png"
    fig.savefig(file_path, dpi=170, bbox_inches="tight")
    plt.close(fig)
    display(IPyImage(filename=str(file_path), width=930))
    print("Saved:", file_path)

# Interactive dropdown for Jupyter; command-line fallback runs Flora alone.
try:
    import ipywidgets as widgets
    _category = widgets.Dropdown(
        options=[("Flora", "flora"), ("Fauna", "fauna"),
                 ("Decorative", "dekoratif"), ("Geometric", "geometris")],
        value="flora", description="Kategori:")
    _grayscale = widgets.Checkbox(value=False, description="Hitam-putih")
    _output = widgets.interactive_output(show_endek_category,
            {"category": _category, "grayscale": _grayscale})
    display(widgets.VBox([widgets.HBox([_category, _grayscale]), _output]))
except ImportError:
    print("ipywidgets not installed; displaying Flora only.")
    show_endek_category(category="flora", grayscale=False)


## 2B. Image preprocessing implemented in the original training code

The recorded loader performs: **RGB conversion → resize to 256×256 → ToTensor → normalization to [-1, 1]**. For training only, the transform adds a random horizontal flip (p = 0.5) and probabilistic ColorJitter (brightness/contrast/saturation ±0.08, hue ±0.02, p = 0.3). Validation and test loaders do not use those stochastic augmentations. The figure below displays an actual Endek source before and after its deterministic preprocessing; it does not imply the random augmentation was applied to this specific preview. No dataset files are overwritten.

In [ ]:
p=sample_paths[0][1]
with Image.open(p) as im:
    original=im.convert("RGB").copy()
resized=original.resize((256,256))
array=np.asarray(resized,dtype=np.float32)/255.0
normalised=(array-.5)/.5
reconstructed=np.uint8(np.clip((normalised*.5+.5)*255,0,255))
print("Source:",p.name,"Original dimensions:",original.size,"Resized:",resized.size)
print("Range after normalization:",float(normalised.min()),float(normalised.max()))
fig,axes=plt.subplots(1,3,figsize=(10,3.5))
axes[0].imshow(original);axes[0].set_title("Original RGB")
axes[1].imshow(resized);axes[1].set_title("Resized to 256×256")
axes[2].imshow(reconstructed);axes[2].set_title("Normalized then reconstructed")
for ax in axes: ax.axis("off")
fig.tight_layout();fig.savefig(FIG/"FIG_01B_PREPROCESSING_PIPELINE.png",dpi=170,bbox_inches="tight");plt.close(fig)
display(IPyImage(filename=str(FIG/"FIG_01B_PREPROCESSING_PIPELINE.png"),width=880))
print("Training augmentation is stochastic and is described separately; its output is not represented by the preview.")

## 3. Preprocessing and structured design-knowledge metadata

Original metadata are loaded as recorded; this section **does not rewrite prompts**. Structured fields come from existing CSVs, and automated descriptions must not be mistaken for expert-verified attributes.

In [ ]:
prompt_fields=["label","motif_source","technique","layout_pattern","composition",
               "color_style","cultural_context","weavability_note","prompt","prompt_origin","expert_verified"]
assert set(prompt_fields).issubset(split["train"].columns)
preview=split["train"].sort_values("filename").groupby("label",sort=False).head(1)[prompt_fields]
display(preview[["label","motif_source","technique","layout_pattern","color_style","prompt_origin","expert_verified"]])
for _,r in preview.iterrows():
    print("\n",r["label"].upper(),":",str(r.prompt)[:450])
recorded=pd.concat(split.values(),ignore_index=True)
print("\nMissingness per prompt field:\n",recorded[prompt_fields].isna().sum().to_string())
print("Expert verified:",recorded.expert_verified.astype(str).str.lower().eq("true").sum(),"/",len(recorded))
preview.to_csv(TAB/"TABLE_02_PROMPT_ATTRIBUTE_EXAMPLES.csv",index=False,encoding="utf-8-sig")

## 4. Unified research framework

The workflow treats all saved experiments as one study. Internal ablations answer RQ1–RQ2; adapted DF-GAN–MiniLM is the separate-architecture comparator for RQ3. The diagram is monochrome to facilitate IEEE paper insertion.

In [ ]:
stages=[
 ("Data curation","370 unique Endek images"),
 ("Fixed split","296 train | 37 val | 37 test"),
 ("Design-knowledge prompts","Category | Hash | Transformer"),
 ("Internal models","Six configurations; 3 seeds"),
 ("External comparator","Adapted DF-GAN–MiniLM; 3 seeds"),
 ("Evaluation","FID | KID | LPIPS | seam gap | images"),
]
fig,ax=plt.subplots(figsize=(14,3.8));ax.set_xlim(0,14);ax.set_ylim(0,3.8);ax.axis("off")
xs=[.15,2.55,4.95,7.35,9.75,12.15]
for i,((head,body),x) in enumerate(zip(stages,xs)):
    rect=plt.Rectangle((x,.92),2.05,1.72,facecolor="white",edgecolor="black",linewidth=1.15)
    ax.add_patch(rect)
    ax.text(x+1.025,2.15,head,ha="center",va="center",fontsize=9.1,fontweight="bold")
    ax.text(x+1.025,1.48,body,ha="center",va="center",fontsize=7.9,wrap=True)
    if i<5:ax.annotate("",xy=(xs[i+1]-.03,1.77),xytext=(x+2.1,1.77),arrowprops={"arrowstyle":"->","color":"black","lw":1.2})
ax.text(7.2,.35,"RQ1: conditioning     |     RQ2: seam-aware ablation     |     RQ3: architecture comparison",ha="center",fontsize=10)
fig.tight_layout();fig.savefig(FIG/"FIG_02_UNIFIED_FRAMEWORK_BW.png",dpi=180,bbox_inches="tight");plt.close(fig)
display(IPyImage(filename=str(FIG/"FIG_02_UNIFIED_FRAMEWORK_BW.png"),width=1050))

## 5. Proposed method and actual training terms

The seam-aware configuration uses an opposing-border loss added to generator hinge adversarial and discriminator-feature-matching terms. The code configuration and saved training logs are the evidence; a diagram labels the special regularizer without claiming it improves motif recognizability.

In [ ]:
config=json.loads((RUNS/"transformer_text_seam_42"/"run_config.json").read_text(encoding="utf-8"))
print({k:config.get(k) for k in ("encoder","model_name","image_size","batch_size","latent_dim",
"fm_lambda","seam_lambda","r1_gamma","r1_every","ema_decay","early_stop_enabled","epochs")})
fig,ax=plt.subplots(figsize=(12.5,3.9));ax.set_xlim(0,12.5);ax.set_ylim(0,3.9);ax.axis("off")
nodes=[
 (0.25,2.3,"Endek prompt\nMiniLM embedding","#eff4fa"),
 (0.25,.55,"Latent noise z\n+ class embedding","#eff4fa"),
 (3.45,1.42,"Conditional\nGenerator G","#e1eef5"),
 (6.4,1.42,"Generated Endek\n256×256 image","#f7f7f7"),
 (9.4,2.35,"Conditional D\n+ feature matching","#f1f1f1"),
 (9.4,.38,"Opposite-border\nL1 seam penalty","#fde1ce")]
for x,y,t,c in nodes:
    b=plt.Rectangle((x,y),2.3,1.02,facecolor=c,edgecolor="black",linewidth=1.15)
    ax.add_patch(b);ax.text(x+1.15,y+.51,t,va="center",ha="center",fontsize=9)
for x0,y0,x1,y1 in [(2.55,2.81,3.45,1.93),(2.55,1.06,3.45,1.93),(5.75,1.93,6.4,1.93),
 (8.7,1.93,9.4,2.86),(8.7,1.93,9.4,.89)]:
    ax.annotate("",xy=(x1,y1),xytext=(x0,y0),arrowprops=dict(arrowstyle="->",lw=1.2,color="black"))
ax.text(6.25,.12,"Proposed element: seam penalty; compare against λseam=0 with matched backbone",fontsize=9,ha="center")
fig.tight_layout();fig.savefig(FIG/"FIG_03_PROPOSED_METHOD_COLOR.png",dpi=175,bbox_inches="tight");plt.close(fig)
display(IPyImage(filename=str(FIG/"FIG_03_PROPOSED_METHOD_COLOR.png"),width=990))

## 6. Reconstruct all 21 completed training runs

The internal study contains **six configurations × three seeds**. The independently adapted DF-GAN–MiniLM contains **three additional runs**. CPU versus CUDA is recorded as an implementation detail, not a model-versus-model comparison.

In [ ]:
internal=pd.read_csv(INTERNAL/"E1_E7_NEW_RESULTS_BY_RUN.csv")
external=pd.read_csv(EXTERNAL/"E8_EXTERNAL_MACRO_BY_RUN.csv")
def display_model(row):
    enc=str(row.encoder); seam=float(row.seam_lambda)>0
    no_class=str(row.disable_class).lower() in ("true","1")
    if enc=="cat":return "Category GAN"
    if enc=="hash_text":return "Hash-Text"
    if seam and no_class:return "Proposed (no class)"
    if seam:return "Proposed Seam-Aware"
    if no_class:return "Transformer (no class)"
    return "Transformer"
internal["model"]=internal.apply(display_model,axis=1)
i=internal.rename(columns={"macro_FID_exploratory":"FID","macro_KID_exploratory":"KID"}).copy()
i["data_source"]="internal saved metrics"
e=external.rename(columns={"macro_FID_exploratory":"FID","macro_KID_exploratory":"KID"}).copy()
e["model"]="DF-GAN–MiniLM (adapted)";e["run"]=e["method"]+"_"+e.seed.astype(str)
e["data_source"]="external saved metrics"
model_runs=pd.concat([i[["run","model","seed","FID","KID","data_source"]],
                       e[["run","model","seed","FID","KID","data_source"]]],ignore_index=True)
assert len(model_runs)==21 and model_runs.model.nunique()==7
assert model_runs.groupby("model").size().eq(3).all()
display(model_runs.sort_values(["model","seed"]))
model_runs.to_csv(TAB/"TABLE_03_ALL_21_RUNS_FID_KID.csv",index=False,encoding="utf-8-sig")

## 7. Implementation settings and checkpoint provenance

Model-specific training settings must not be presented as identical. For reproducibility, inspect the per-run JSON rather than merging different architectures into one unspecified training procedure.

In [ ]:
configs=[]
for path in sorted(RUNS.glob("*/run_config.json")):
    r=json.loads(path.read_text(encoding="utf-8"))
    configs.append({"run":path.parent.name,"seed":r.get("seed"),"encoder":r.get("encoder"),
      "device":r.get("device"),"batch_size":r.get("batch_size"),"image_size":r.get("image_size"),
      "latent_dim":r.get("latent_dim"),"lr_G":r.get("lr_g"),"lr_D":r.get("lr_d"),
      "seam_lambda":r.get("seam_lambda"),"fm_lambda":r.get("fm_lambda"),
      "disable_class":r.get("disable_class"),"early_stop":r.get("early_stop_enabled")})
cfg_frame=pd.DataFrame(configs)
assert len(cfg_frame)==18
display(cfg_frame.head(9))
cfg_frame.to_csv(TAB/"TABLE_04_INTERNAL_TRAINING_CONFIGURATIONS.csv",index=False)
external_checkpoints=[]
for seed in SEEDS:
    run_dir=EXTERNAL/"training_dfgan_minilm_adapted"/("seed_2025_recovery" if seed==2025 else f"seed_{seed}")
    fp=run_dir/"TRAINING_COMPLETE.json"
    assert fp.is_file(),fp
    j=json.loads(fp.read_text(encoding="utf-8"))
    external_checkpoints.append({"seed":seed,"status":j.get("status"),"best_epoch":j.get("best_epoch"),
      "last_epoch":j.get("last_epoch"),"text_encoder":j.get("text_encoder"),
      "checkpoint_exists":Path(j.get("checkpoint")).is_file()})
display(pd.DataFrame(external_checkpoints))
assert all(x["status"]=="COMPLETE" for x in external_checkpoints)
pd.DataFrame(external_checkpoints).to_csv(TAB/"TABLE_05_EXTERNAL_CHECKPOINTS.csv",index=False)

## 8. Training traces and validation-based selection

Curves below use actual saved training logs. They are diagnostic, not a proof of superior generalization. Early-stopping configuration is read from the run JSON.

In [ ]:
fig,ax=plt.subplots(figsize=(9.2,4.1))
for name in ("transformer_text_42","transformer_text_seam_42"):
    f=RUNS/name/"training_log.csv"
    trace=pd.read_csv(f)
    ax.plot(trace.epoch,trace.g_loss,label=name,linewidth=1.25)
ax.set(title="Generator objective during training — seed 42",xlabel="Epoch",ylabel="Recorded generator loss")
ax.grid(alpha=.25);ax.legend(fontsize=8);fig.tight_layout()
fig.savefig(FIG/"FIG_04_TRAINING_LOSS_SEED42.png",dpi=170,bbox_inches="tight");plt.close(fig)
display(IPyImage(filename=str(FIG/"FIG_04_TRAINING_LOSS_SEED42.png"),width=880))
print("Warning: generator objectives differ because seam penalty contributes to one loss.")

## 9. Three-seed FID and KID overview (exploratory)

Scores are read from archived run-level files; they are not recomputed. Because the test set has only 37 real images, macro FID/KID are **exploratory**. The adapted DF-GAN–MiniLM differs in architecture and training settings.

In [ ]:
model_summary=model_runs.groupby("model",as_index=False).agg(
    mean_FID=("FID","mean"),sd_FID=("FID","std"),mean_KID=("KID","mean"),sd_KID=("KID","std"))
model_summary=model_summary.sort_values("mean_FID")
display(model_summary.round(5))
model_summary.to_csv(TAB/"TABLE_06_FID_KID_THREE_SEEDS.csv",index=False)
for metric,label,file in [("mean_FID","Macro FID (lower is better)","FIG_05_MACRO_FID.png"),
                          ("mean_KID","Macro KID (lower is better)","FIG_06_MACRO_KID.png")]:
    fig,ax=plt.subplots(figsize=(9.5,4))
    ax.barh(model_summary.model,model_summary[metric],facecolor="white",edgecolor="black")
    ax.invert_yaxis();ax.set_xlabel(label);ax.set_title("Three-seed comparison | exploratory test population (n=37)")
    ax.grid(axis="x",alpha=.2);fig.tight_layout();fig.savefig(FIG/file,dpi=170,bbox_inches="tight");plt.close(fig)
display(IPyImage(filename=str(FIG/"FIG_05_MACRO_FID.png"),width=850))
display(IPyImage(filename=str(FIG/"FIG_06_MACRO_KID.png"),width=850))

## 10. LPIPS perceptual variability and original seam-gap summaries

Original LPIPS records are **perceptual variability among generated images**, not prompt fidelity. Original E7 seam gap is the experiment's opposing-edge diagnostic. Different population/normalization protocols must not be silently mixed.

In [ ]:
lpips=[]
seams=[]
for folder in sorted(RUNS.iterdir()):
    if not folder.is_dir():continue
    eval_dir=folder/"evaluation"
    for typ,filename,target in [("LPIPS","E2_LPIPS.csv",lpips),
                                ("seam_gap","E7_seam_gap_summary.csv",seams)]:
        f=eval_dir/filename
        if f.is_file():
            rows=pd.read_csv(f)
            rows["run"]=folder.name
            target.append(rows)
lpips_df=pd.concat(lpips,ignore_index=True) if lpips else pd.DataFrame()
seams_df=pd.concat(seams,ignore_index=True) if seams else pd.DataFrame()
print("Original LPIPS rows:",len(lpips_df),"| Original seam-gap rows:",len(seams_df))
if not lpips_df.empty:
    display(lpips_df.head(8))
    lpips_df.to_csv(TAB/"TABLE_07_ORIGINAL_LPIPS_BY_CLASS.csv",index=False)
if not seams_df.empty:
    display(seams_df.head(8))
    seams_df.to_csv(TAB/"TABLE_08_ORIGINAL_SEAM_BY_CLASS.csv",index=False)

## 11. Full image-level diagnostic (3 seeds, seven models)

Image-level CSVs have already been produced from the saved PNGs. **7 models × 4 classes × 128 generated images × 3 seeds = 10,752** rows. Opposing-edge seam gap is a pixel-level diagnostic; image-detail and periodicity proxies are descriptive and may reward blur.

In [ ]:
evidence=[]
for seed in SEEDS:
    f=ADD/f"SEED_{seed}"/"EVAL_IMAGE_LEVEL.csv"
    assert f.is_file(),f
    t=pd.read_csv(f);assert len(t)==3584,(seed,len(t))
    evidence.append(t)
image_level=pd.concat(evidence,ignore_index=True)
assert len(image_level)==10752
print("Total image-level observations:",len(image_level))
print("Model labels:",sorted(image_level.model.unique()))
d=image_level.groupby(["model","seed"],as_index=False).agg(
    seam_gap=("seam_gap","mean"),detail_energy=("detail_energy","mean"),
    periodicity_proxy=("periodicity_proxy","mean"),n=("image_path","size"))
display(d.round(5))
d.to_csv(TAB/"TABLE_09_IMAGE_DIAGNOSTICS_BY_SEED.csv",index=False,encoding="utf-8-sig")
fig,ax=plt.subplots(figsize=(9.5,4))
focus=d[d.model.isin(["Transformer","Proposed Seam-Aware"])]
for model,part in focus.groupby("model"):
    ax.plot(part.seed.astype(str),part.seam_gap,marker="o",label=model)
ax.set(xlabel="Training seed",ylabel="Mean opposing-edge seam gap (lower is better)",
       title="Matched backbone: standard Transformer vs seam-aware")
ax.legend();ax.grid(alpha=.25);fig.tight_layout()
fig.savefig(FIG/"FIG_07_SEAM_GAP_SEEDS.png",dpi=175,bbox_inches="tight");plt.close(fig)
display(IPyImage(filename=str(FIG/"FIG_07_SEAM_GAP_SEEDS.png"),width=900))

## 12. Actual original image versus all generated configurations

The archived model-comparison panels show **real Endek test images** beside six internal variants and an adapted DF-GAN–MiniLM baseline. Internal outputs are selected by an indexed source/prompt relation. The external model receives the same indexed prompt but not an identical latent vector. Images are generated results, not illustrative simulations.

In [ ]:
for seed in SEEDS:
    src=ADD/f"SEED_{seed}"/"FIG_MODEL_COMPARISON.png"
    manifest=ADD/f"SEED_{seed}"/"FIG_INPUT_MANIFEST.csv"
    assert src.is_file() and manifest.is_file()
    m=pd.read_csv(manifest)
    assert {"model","seed","test_source","prompt","image_path"}.issubset(m.columns)
    assert m["model"].nunique()==7
    dst=FIG/f"FIG_08_IMAGE_MODEL_COMPARISON_SEED_{seed}.png"
    shutil.copy2(src,dst)
    print("SEED",seed,"| manifest rows",len(m),"| reference:",manifest)
    display(IPyImage(filename=str(dst),width=1150))

## 13. Tiled 2×2 images: proposed versus standard Transformer and DF-GAN

Repetition is used to inspect seam discontinuity; **tileability is not weaving readiness**. Dashed reference lines in these archived figures mark tile junctions.

In [ ]:
for seed in SEEDS:
    src=ADD/f"SEED_{seed}"/"FIG_SEAM_TILE_COMPARISON.png"
    assert src.is_file()
    dst=FIG/f"FIG_09_TILE_COMPARISON_SEED_{seed}.png"
    shutil.copy2(src,dst)
    print("Tile figure, seed",seed)
    display(IPyImage(filename=str(dst),width=1000))

## 14. Multilingual CLIP diagnostic — partial evidence only

Only seed **42** has a saved CLIP cosine diagnostic. No conclusion for all three seeds should be drawn from it. Prompt descriptions were algorithmically constructed and not checked image-by-image by Endek experts.

In [ ]:
clip_paths={seed:ADD/f"SEED_{seed}"/"EVAL_MULTILINGUAL_CLIP_DIAGNOSTIC.csv" for seed in SEEDS}
availability={seed:path.is_file() for seed,path in clip_paths.items()}
print("Saved CLIP data availability:",availability)
if availability[42]:
    clip=pd.read_csv(clip_paths[42])
    assert len(clip)==448
    cols=["model","motif_class","multilingual_clip_cosine_diagnostic"]
    clip_avg=clip.groupby("model",as_index=False)["multilingual_clip_cosine_diagnostic"].agg(["mean","std","count"])
    display(clip_avg.round(5))
    clip_avg.to_csv(TAB/"TABLE_10_CLIP_DIAGNOSTIC_SEED42_ONLY.csv")
else:
    print("No CLIP evaluation scores are available: do not invent values.")

## 15. Expert validation and incomplete assessments

The existing blinded forms are templates, not responses. Expert scores, motif recognizability, cultural appropriateness, and weaving feasibility cannot be reported unless completed expert forms exist.

In [ ]:
forms=[]
for seed in SEEDS:
    f=ADD/f"SEED_{seed}"/"EXPERT_BLIND_RATING_BLANK.csv"
    if f.is_file():
        t=pd.read_csv(f); forms.append((seed,len(t),t.notna().sum().to_dict()))
print("Blinded image-form status (seed, rows, populated fields):")
for seed,n,col in forms:
    print(seed,n, {k:v for k,v in col.items() if k.endswith("_1to5") and v>0})
print("Warning: any score counts above zero would require genuine expert response records; original forms are blank.")
print("Do not circulate EXPERT_PRIVATE_MAPPING_DO_NOT_SHARE.csv to evaluators.")

## 15B. Archive-wide experiment completeness (including unfinished workflows)

Beyond the 21 paper-comparable runs, the archive includes **counterfactual color diagnostics, nearest-neighbor feature diagnostics, an expert-rating protocol, and a StyleGAN2-ADA data export**. Their existence does not imply validated expert ratings or completed StyleGAN2 training. Earlier pilot/smoke folders are inventoried separately and are **not combined** into the final comparable run totals.

In [ ]:
counterfactual=list(RUNS.glob("*/counterfactual/*/E3_counterfactual_manifest.csv"))
nn=list(RUNS.glob("*/evaluation/E2_NN_feature_diagnostic.csv"))
expert_form=INTERNAL/"E4_blinded_expert_evaluation"/"E4_EVALUATION_FORM_BLANK.csv"
style_protocol=json.loads((INTERNAL/"E6_StyleGAN2ADA_export"/"E6_export_protocol.json").read_text(encoding="utf-8"))
external_status=json.loads((EXTERNAL/"TRAINING_STATUS.json").read_text(encoding="utf-8"))
expert=pd.read_csv(expert_form) if expert_form.is_file() else pd.DataFrame()
ratings=[c for c in expert.columns if c.endswith("_1to5")]
expert_scores=int(expert[ratings].notna().sum().sum()) if ratings else 0
inventory=pd.DataFrame([
 {"component":"Core internal conditioning/ablation runs","evidence":str(INTERNAL/"E1_E7_NEW_RESULTS_BY_RUN.csv"),"records":len(internal),"status":"COMPLETED"},
 {"component":"Nearest-neighbor feature diagnostics","evidence":"*/evaluation/E2_NN_feature_diagnostic.csv","records":len(nn),"status":"ARCHIVED DIAGNOSTIC"},
 {"component":"Counterfactual motif/color diagnostics","evidence":"*/counterfactual/*/E3_counterfactual_manifest.csv","records":len(counterfactual),"status":"ARCHIVED DIAGNOSTIC"},
 {"component":"Blinded expert ratings","evidence":str(expert_form),"records":len(expert),"status":"FORM PREPARED; SCORES="+str(expert_scores)},
 {"component":"StyleGAN2-ADA","evidence":str(INTERNAL/"E6_StyleGAN2ADA_export"/"E6_export_protocol.json"),"records":0,"status":style_protocol["status"]},
 {"component":"Adapted DF-GAN–MiniLM","evidence":str(EXTERNAL/"TRAINING_STATUS.json"),"records":len(external),"status":external_status.get("DF-GAN–MiniLM (adapted)","SEE JSON")},
 {"component":"Early pilot/smoke artifacts","evidence":str(ROOT/"EndekGAN_IEEE_V3_OUTPUT"),"records":0,"status":"SEPARATE PILOT; NOT POOLED"}
])
display(inventory)
inventory.to_csv(TAB/"TABLE_12_EXPERIMENT_COMPLETENESS.csv",index=False)
print("Important: no human expert rating is established by blank forms; StyleGAN2-ADA was NOT trained.")


## 16. Evidence map for the article's RQ1–RQ3

This table identifies which archived experiments can support each research question and what they **cannot** establish.

In [ ]:
rq=pd.DataFrame([
 {"RQ":"RQ1","Claim tested":"Category vs lexical hash vs multilingual Transformer conditioning",
  "Data available":"Internal FID/KID, LPIPS, image panels across three seeds",
  "Remaining limit":"No expert-verified per-attribute text-image agreement"},
 {"RQ":"RQ2","Claim tested":"Opposing-border L1 regularization and class-embedding ablations",
  "Data available":"Matched transformer/seam runs, seam-gap, FID/KID, LPIPS, 2×2 tiles",
  "Remaining limit":"Low seam gap does not guarantee ornament recognizability or weavability"},
 {"RQ":"RQ3","Claim tested":"Internal GAN configurations vs adapted DF-GAN–MiniLM",
  "Data available":"21 run metrics, three-seed images, archived diagnostics",
  "Remaining limit":"Different architectures/training; descriptive comparison only"},
])
display(rq)
rq.to_csv(TAB/"TABLE_11_RQ_EVIDENCE_MAP.csv",index=False)

## 17. Audit manifest, limitations, and handoff

This notebook is a **reproducible review of existing evidence**. It records inputs and saves paper-ready derivative tables and images in a new folder. No checkpoint or training CSV outside this folder is changed.

In [ ]:
sources={
 "dataset_audit":SPLITS/"dataset_audit.json",
 "fixed_train":SPLITS/"train_split.csv",
 "fixed_validation":SPLITS/"val_split.csv",
 "fixed_test":SPLITS/"test_split.csv",
 "internal_run_metrics":INTERNAL/"E1_E7_NEW_RESULTS_BY_RUN.csv",
 "external_run_metrics":EXTERNAL/"E8_EXTERNAL_MACRO_BY_RUN.csv",
 "full_diagnostic_seed42":ADD/"SEED_42"/"EVAL_IMAGE_LEVEL.csv",
 "full_diagnostic_seed123":ADD/"SEED_123"/"EVAL_IMAGE_LEVEL.csv",
 "full_diagnostic_seed2025":ADD/"SEED_2025"/"EVAL_IMAGE_LEVEL.csv",
 "semantic_seed42":clip_paths[42],
}
manifest={k:{"path":str(p),"exists":p.is_file(),"bytes":p.stat().st_size if p.is_file() else None}
          for k,p in sources.items()}
meta={"study":"EndekGAN-T2I: one integrated research experiment",
      "training_runs":21,"internal_runs":18,"external_runs":3,
      "fixed_split":{"train":296,"validation":37,"test":37},
      "image_level_evaluations":len(image_level),
      "clip_diagnostic_seeds":[s for s,a in availability.items() if a],
      "expert_verified_prompt_rows":int(audit["expert_verified_rows"]),
      "source_manifest":manifest,
      "limitations":["FID/KID exploratory on 37 real test images",
       "CLIP only saved for seed 42; caption metadata not expert-verified",
       "Seam metric measures pixel border agreement, not motif quality",
       "DF-GAN adapted and independently trained; different protocol"] }
(HERE/"SOURCE_AUDIT_MANIFEST.json").write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding="utf-8")
assert len(list(FIG.glob("*.png")))>=12
assert len(list(TAB.glob("*.csv")))>=10
print("Notebook completed; FIGURES:",len(list(FIG.glob("*.png"))),"TABLES:",len(list(TAB.glob("*.csv"))))
print("Source manifest:",HERE/"SOURCE_AUDIT_MANIFEST.json")
print("No training has been executed.")